In [1]:
#All the imports
import h5py
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d
from scipy.stats import norm
from scipy.stats import kurtosis
from scipy.ndimage import maximum_filter, label
import multiprocessing as mp
import os
from scipy.signal import find_peaks

In [2]:
#Function Definitions
def partial_x_2d(f,dx):
    """Compute ∂f/∂x using 2nd-order central differences with periodic BCs."""
    Nx, Ny = f.shape
    df_dx = np.zeros_like(f)
    
    # Central difference in x-direction (axis=0)
    df_dx[1:Nx-1, :] = (f[2:Nx, :] - f[0:Nx-2, :]) / (2 * dx)
    
    # Periodic boundaries (x-direction)
    df_dx[0, :] = (f[1, :] - f[-1, :]) / (2 * dx)
    df_dx[-1, :] = (f[0, :] - f[-2, :]) / (2 * dx)
    
    return df_dx

def partial_y_2d(f,dy):
    """Compute ∂f/∂y using 2nd-order central differences with periodic BCs."""
    Nx, Ny = f.shape
    df_dy = np.zeros_like(f)
    
    # Central difference in y-direction (axis=1)
    df_dy[:, 1:Ny-1] = (f[:, 2:Ny] - f[:, 0:Ny-2]) / (2 * dy)
    
    # Periodic boundaries (y-direction)
    df_dy[:, 0] = (f[:, 1] - f[:, -1]) / (2 * dy)
    df_dy[:, -1] = (f[:, 0] - f[:, -2]) / (2 * dy)
    
    return df_dy

def partial_x_fft(f, dx):
    """
    Calculate ∂f/∂x using FFT on a 2D grid.

    Parameters:
        f (2D np.array): Function values on a 2D grid (shape: Nx × Ny).
        dx (float): Grid spacing in the x-direction.

    Returns:
        df_dx (2D np.array): Partial derivative ∂f/∂x.
    """
    nx, ny = f.shape
    kx = np.fft.fftfreq(nx, d=dx) * 2 * np.pi  # Wavenumbers in x
    kx = kx[:, np.newaxis]                     # Make it 2D for broadcasting

    f_hat = np.fft.fft2(f)
    df_dx_hat = 1j * kx * f_hat
    df_dx = np.fft.ifft2(df_dx_hat).real

    return np.array(df_dx)

def partial_y_fft(f, dy):
    """
    Calculate ∂f/∂y using FFT on a 2D grid.

    Parameters:
        f (2D np.array): Function values on a 2D grid (shape: Nx × Ny).
        dy (float): Grid spacing in the y-direction.

    Returns:
        df_dy (2D np.array): Partial derivative ∂f/∂y.
    """
    nx, ny = f.shape
    ky = np.fft.fftfreq(ny, d=dy) * 2 * np.pi  # Wavenumbers in y
    ky = ky[np.newaxis, :]                     # Make it 2D for broadcasting

    f_hat = np.fft.fft2(f)
    df_dy_hat = 1j * ky * f_hat
    df_dy = np.fft.ifft2(df_dy_hat).real

    return df_dy

def index_of_just_smaller(arr, value):
    arr = np.asarray(arr)
    mask = arr < value
    if not np.any(mask):
        return None  # No smaller value exists
    return np.argmax(np.where(mask, arr, -np.inf))

def compute_radial_spectrum(fld, dx, dy, dtype=float):
    nx, ny = fld.shape

    # Fourier wavenumbers
    kx = np.fft.fftfreq(nx, d=dx) * 2 * np.pi
    ky = np.fft.fftfreq(ny, d=dy) * 2 * np.pi
    KX, KY = np.meshgrid(kx, ky, indexing='ij')
    K_perp = np.sqrt(KX**2 + KY**2)

    # Power spectrum density
    ff = np.abs(np.fft.ifft2(fld))**2   # use fft2, not ifft2
    P_flat = ff.ravel()
    K_flat = K_perp.ravel()

    # Radial bins
    dk = kx[1] - kx[0]
    k_bins = np.arange(0, np.max(K_flat) + dk, dk)
    nk = len(k_bins)
    P_k = np.zeros(nk, dtype=dtype)

    # Bin indices (vectorized)
    idx_float = K_flat / dk
    i1 = np.floor(idx_float).astype(int)
    i2 = i1 + 1
    w = idx_float - i1   # weight to upper bin

    # Accumulate into bins
    for i in range(len(K_flat)):
        if i1[i] >= 0 and i2[i] < nk:
            P_k[i1[i]] += P_flat[i] * (1 - w[i])
            P_k[i2[i]] += P_flat[i] * w[i]

    # Pack into recarray
    tags = ['kk', 'sp']
    sps = np.recarray((nk,), dtype=[(t, dtype) for t in tags])
    sps['kk'] = k_bins
    sps['sp'] = P_k / dk

    return sps

In [4]:
# List of simulation directories
base_root = "/home/UNN/w24021992/NAS/simulations_beta_scan"

sim_dirs = [

    # f"{base_root}/beta_p_0.0625/beta_e_0.25",
    # f"{base_root}/beta_p_0.0625/beta_e_1",
    # f"{base_root}/beta_p_0.0625/beta_e_4",

    # f"{base_root}/beta_p_0.25/beta_e_1",
    # f"{base_root}/beta_p_0.25/beta_e_4",
    # f"{base_root}/beta_p_0.25/beta_e_16",

    # f"{base_root}/beta_p_1/beta_e_0.0625",
    # f"{base_root}/beta_p_1/beta_e_0.25",
    f"{base_root}/beta_p_1/beta_e_1",
    # f"{base_root}/beta_p_1/beta_e_4",
    # f"{base_root}/beta_p_1/beta_e_16",

    # f"{base_root}/beta_p_4/beta_e_0.0625",
    # f"{base_root}/beta_p_4/beta_e_0.25",
    # f"{base_root}/beta_p_4/beta_e_1",
    # # f"{base_root}/beta_p_4/beta_e_4",
    # f"{base_root}/beta_p_4/beta_e_16",

    # f"{base_root}/beta_p_16/beta_e_0.0625",
    # f"{base_root}/beta_p_16/beta_e_0.25",
    # f"{base_root}/beta_p_16/beta_e_16",
]

dx = 0.0625
dy = 0.0625

In [5]:
times = np.arange(0, 301)   # t = 0 to 300

for sim_dir in sim_dirs:

    print(f"\nProcessing {sim_dir}")

    Jmax = []

    # ---------- Compute Jmax(t) ----------
    for t in times:
        File_Bx = f"{sim_dir}/Bx_ApJ_t{t}.h5"
        File_By = f"{sim_dir}/By_ApJ_t{t}.h5"
        File_Bz = f"{sim_dir}/Bz_ApJ_t{t}.h5"

        with h5py.File(File_Bx, 'r') as fBx, \
             h5py.File(File_By, 'r') as fBy, \
             h5py.File(File_Bz, 'r') as fBz:

            Bx = fBx['DS1'][:].T
            By = fBy['DS1'][:].T
            Bz = fBz['DS1'][:].T

        Jx =  partial_y_fft(Bz, dx)
        Jy = -partial_x_fft(Bz, dx)
        Jz =  partial_x_fft(By, dx) - partial_y_fft(Bx, dx)

        mod_J = np.sqrt(Jx**2 + Jy**2 + Jz**2)
        Jmax.append(np.max(mod_J))

    Jmax = np.array(Jmax)

    # ---------- Peak finding (your logic) ----------
    peaks, props = find_peaks(
        Jmax,
        prominence=0.02 * np.max(Jmax)
    )

    if len(peaks) == 0:
        print("  No significant peak found — skipping plot")
        continue

    first_good_peak = peaks[0]
    t_peak = times[first_good_peak]
    J_peak = Jmax[first_good_peak]

    print(f"  First significant peak: t = {t_peak}, Jmax = {J_peak:.3e}")

    # ---------- Plot ----------
    plots_dir = os.path.join(sim_dir, "plots")
    os.makedirs(plots_dir, exist_ok=True)

    plt.figure(figsize=(7, 5))
    plt.plot(times, Jmax, lw=2, label=r'$\max(|\mathbf{J}|)$')

    plt.axvline(t_peak, linestyle='--', linewidth=1.5)
    plt.scatter(t_peak, J_peak, zorder=5)

    plt.text(
        t_peak,
        J_peak,
        f'  Peak\n  t = {t_peak}\n  $|J|_{{max}}$ = {J_peak:.3e}',
        verticalalignment='bottom',
        horizontalalignment='left',
        bbox=dict(boxstyle="round", alpha=0.25)
    )

    plt.xlabel("Time")
    plt.ylabel(r"$\max(|\mathbf{J}|)$")
    plt.title(f"Maximum current density vs time")
    plt.tight_layout()

    save_path = os.path.join(plots_dir, "Reconnection_Time.png")
    plt.savefig(save_path, dpi=300)
    plt.close()

    print(f"  Saved: {save_path}")


Processing /home/UNN/w24021992/NAS/simulations_beta_scan/beta_p_1/beta_e_1
  First significant peak: t = 40, Jmax = 5.519e+00
  Saved: /home/UNN/w24021992/NAS/simulations_beta_scan/beta_p_1/beta_e_1/plots/Reconnection_Time.png


In [ ]:

# peaks, props = find_peaks(
#     Jmax,
#     prominence=0.02 * np.max(Jmax)
# )

# if len(peaks) == 0:
#     print("  No significant peak found — skipping plot")


# first_good_peak = peaks[0]
# t_peak = times[first_good_peak]
# J_peak = Jmax[first_good_peak]

# print(f"  First significant peak: t = {t_peak}, Jmax = {J_peak:.3e}")

# # ---------- Plot ----------
# plots_dir = os.path.join(sim_dir, "plots")
# os.makedirs(plots_dir, exist_ok=True)

# plt.figure(figsize=(7, 5))
# plt.plot(times, Jmax, lw=2, label=r'$\max(|\mathbf{J}|)$')

# plt.axvline(t_peak, linestyle='--', linewidth=1.5)
# plt.scatter(t_peak, J_peak, zorder=5)

# plt.text(
#     t_peak,
#     J_peak,
#     f'  Peak\n  t = {t_peak}\n  $|J|_{{max}}$ = {J_peak:.3e}',
#     verticalalignment='bottom',
#     horizontalalignment='left',
#     bbox=dict(boxstyle="round", alpha=0.25)
# )

# plt.xlabel("Time step")
# plt.ylabel(r"$\max(|\mathbf{J}|)$")
# plt.title(f"Maximum current density vs time\n{sim_dir.split('/')[-1]}")
# plt.grid(True)
# plt.tight_layout()

# save_path = os.path.join(plots_dir, "Reconnection_Time.png")
# plt.savefig(save_path, dpi=300)
# plt.close()

# print(f"  Saved: {save_path}")